# API and Web Scraping: Solutions

Instructor copy. Run the setup cell first (it starts the practice API), then any
solution below.

In [ ]:
# SETUP: start the practice API
import json, threading, time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import urlparse, parse_qs

CHARACTERS = [
    {"id": i, "name": name, "status": status,
     "species": "Human" if i % 3 else "Alien",
     "origin": {"name": origin, "url": f"http://localhost/location/{i}"},
     "episode": [f"http://localhost/episode/{e}" for e in range(1, (i % 4) + 2)]}
    for i, (name, status, origin) in enumerate([
        ("Rick Sanchez", "Alive", "Earth C-137"), ("Morty Smith", "Alive", "Earth C-137"),
        ("Summer Smith", "Alive", "Earth"), ("Beth Smith", "Alive", "Earth"),
        ("Jerry Smith", "Alive", "Earth"), ("Abadango Cluster", "unknown", "Abadango"),
        ("Abradolf Lincler", "unknown", "Earth"), ("Adjudicator Rick", "Dead", "unknown"),
        ("Agency Director", "Dead", "Earth"), ("Alan Rails", "Dead", "unknown"),
        ("Albert Einstein", "Dead", "Earth"), ("Alexander", "Dead", "Egypt"),
        ("Alien Googah", "unknown", "unknown"), ("Alien Morty", "unknown", "unknown"),
        ("Alien Rick", "unknown", "unknown"), ("Amish Cyborg", "Dead", "Earth"),
        ("Annie", "Alive", "Earth"), ("Antenna Morty", "Alive", "unknown"),
        ("Antenna Rick", "unknown", "unknown"), ("Ants in my Eyes Johnson", "unknown", "Earth"),
    ], start=1)
]

SHOP_HTML = '''<!doctype html><html><head><title>DEC Practice Shop</title></head><body>
<h1 id="page-title">Today's Products</h1><p class="intro">A small page to practise scraping.</p>
<div class="product" data-sku="LP-001"><h2 class="name">Laptop Pro</h2>
<span class="price">450000</span><span class="stock in-stock">In stock</span>
<a href="/product/lp-001">details</a></div>
<div class="product" data-sku="PH-002"><h2 class="name">Phone X</h2>
<span class="price">180000</span><span class="stock out-of-stock">Out of stock</span>
<a href="/product/ph-002">details</a></div>
<div class="product" data-sku="MS-003"><h2 class="name">Wireless Mouse</h2>
<span class="price">8000</span><span class="stock in-stock">In stock</span>
<a href="/product/ms-003">details</a></div>
<table id="sales"><thead><tr><th>Month</th><th>Units</th><th>Revenue</th></tr></thead>
<tbody><tr><td>JAN</td><td>340</td><td>1200000</td></tr>
<tr><td>FEB</td><td>318</td><td>980000</td></tr>
<tr><td>MAR</td><td>362</td><td>1450000</td></tr></tbody></table>
<ul class="links"><li><a href="/about">About</a></li>
<li><a href="https://example.com/external">External site</a></li>
<li><a href="/contact">Contact</a></li></ul></body></html>'''

ROBOTS = "User-agent: *\nDisallow: /admin\nDisallow: /checkout\nCrawl-delay: 1\n"
_state = {"flaky_calls": 0, "limited_calls": 0}
PAGE_SIZE = 6


class Handler(BaseHTTPRequestHandler):
    def log_message(self, *args):
        pass

    def _send(self, code, payload, content_type="application/json", extra_headers=None):
        body = payload if isinstance(payload, bytes) else json.dumps(payload).encode()
        self.send_response(code)
        self.send_header("Content-Type", content_type)
        self.send_header("Content-Length", str(len(body)))
        for k, v in (extra_headers or {}).items():
            self.send_header(k, v)
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        parsed = urlparse(self.path)
        query = parse_qs(parsed.query)
        path = parsed.path
        if path == "/api/character":
            page = int(query.get("page", ["1"])[0])
            status_filter = query.get("status", [None])[0]
            rows = CHARACTERS
            if status_filter:
                rows = [c for c in rows if c["status"].lower() == status_filter.lower()]
            pages = max(1, -(-len(rows) // PAGE_SIZE))
            if page < 1 or page > pages:
                return self._send(404, {"error": "There is nothing here"})
            start = (page - 1) * PAGE_SIZE
            base = f"{BASE_URL}/api/character"
            suffix = f"&status={status_filter}" if status_filter else ""
            return self._send(200, {
                "info": {"count": len(rows), "pages": pages,
                         "next": f"{base}?page={page+1}{suffix}" if page < pages else None,
                         "prev": f"{base}?page={page-1}{suffix}" if page > 1 else None},
                "results": rows[start:start + PAGE_SIZE]})
        if path == "/api/slow":
            time.sleep(float(query.get("seconds", ["2"])[0]))
            return self._send(200, {"ok": True})
        if path == "/api/flaky":
            _state["flaky_calls"] += 1
            if _state["flaky_calls"] % 3 != 0:
                return self._send(503, {"error": "service unavailable"})
            return self._send(200, {"ok": True, "call": _state["flaky_calls"]})
        if path == "/api/limited":
            _state["limited_calls"] += 1
            if _state["limited_calls"] % 4 != 0:
                return self._send(429, {"error": "too many requests"},
                                  extra_headers={"Retry-After": "1"})
            return self._send(200, {"ok": True, "call": _state["limited_calls"]})
        if path == "/api/protected":
            if self.headers.get("X-API-Key") != "dec-secret-key":
                return self._send(401, {"error": "missing or invalid API key"})
            return self._send(200, {"secret": "you are authorised", "plan": "pro"})
        if path == "/api/broken":
            return self._send(200, b"this is not json at all", content_type="text/plain")
        if path == "/shop.html":
            return self._send(200, SHOP_HTML.encode(), content_type="text/html")
        if path == "/robots.txt":
            return self._send(200, ROBOTS.encode(), content_type="text/plain")
        return self._send(404, {"error": "not found"})

    def do_POST(self):
        length = int(self.headers.get("Content-Length", 0))
        raw = self.rfile.read(length)
        try:
            payload = json.loads(raw) if raw else {}
        except json.JSONDecodeError:
            return self._send(400, {"error": "body was not valid json"})
        if urlparse(self.path).path != "/api/orders":
            return self._send(404, {"error": "not found"})
        if "item" not in payload:
            return self._send(422, {"error": "field 'item' is required"})
        return self._send(201, {"created": True, "order_id": 9001, "echo": payload})


class QuietServer(ThreadingHTTPServer):
    daemon_threads = True

    def handle_error(self, request, client_address):
        pass


_server = QuietServer(("127.0.0.1", 0), Handler)
threading.Thread(target=_server.serve_forever, daemon=True).start()
BASE_URL = f"http://127.0.0.1:{_server.server_address[1]}"
time.sleep(0.3)
print(f"practice API at {BASE_URL}")

## Response object: tasks

In [ ]:
# Task 1
import requests

response = requests.get(f"{BASE_URL}/api/character", params={"page": 3}, timeout=5)
print(f"status: {response.status_code}")
print(f"url:    {response.url}")
print(f"results: {len(response.json()['results'])}")

In [ ]:
# Task 2
try:
    requests.get(f"{BASE_URL}/api/slow", params={"seconds": 2}, timeout=1)
except requests.exceptions.Timeout:
    print("timed out at 1 second")

response = requests.get(f"{BASE_URL}/api/slow", params={"seconds": 1}, timeout=5)
print(f"succeeded in {response.elapsed.total_seconds():.2f}s")

In [ ]:
# Task 3
response = requests.get(f"{BASE_URL}/api/character", params={"page": 99}, timeout=5)
try:
    response.raise_for_status()
except requests.exceptions.HTTPError as e:
    print(f"HTTPError: {e}")
    print(f"body: {response.json()}")

In [ ]:
# Task 4
response = requests.get(f"{BASE_URL}/api/broken", timeout=5)
print(f"status {response.status_code}, content-type {response.headers['Content-Type']}")
try:
    response.json()
except requests.exceptions.JSONDecodeError as e:
    print(f"JSONDecodeError: {str(e)[:60]}")
print(f"the body was: {response.text!r}")

In [ ]:
# DE Task 5: safe_get returning (data, error)
def safe_get(url, params=None, timeout=5):
    """Return (data, error). error is None on success."""
    try:
        response = requests.get(url, params=params, timeout=timeout)
        response.raise_for_status()
        return response.json(), None
    except requests.exceptions.Timeout:
        return None, "timeout"
    except requests.exceptions.ConnectionError:
        return None, "connection failed"
    except requests.exceptions.HTTPError as e:
        return None, f"http {e.response.status_code}"
    except requests.exceptions.JSONDecodeError:
        return None, "response was not json"
    except requests.exceptions.RequestException as e:
        return None, type(e).__name__


for label, url, params in [
    ("good",        f"{BASE_URL}/api/character", {"page": 1}),
    ("bad page",    f"{BASE_URL}/api/character", {"page": 99}),
    ("not json",    f"{BASE_URL}/api/broken", None),
    ("no server",   "http://127.0.0.1:9/nope", None),
]:
    data, error = safe_get(url, params)
    print(f"{label:<10} error={error!s:<22} data={'yes' if data else 'no'}")

## Pagination, retries, rate limits: tasks

In [ ]:
# Task 1: follow next
collected = []
url = f"{BASE_URL}/api/character"
while url:
    payload = requests.get(url, timeout=5).json()
    collected.extend(payload["results"])
    url = payload["info"]["next"]

print(f"records: {len(collected)}, distinct ids: {len({c['id'] for c in collected})}")

In [ ]:
# Task 2: range with the CORRECT parameter name
first = requests.get(f"{BASE_URL}/api/character", timeout=5).json()
pages = first["info"]["pages"]

by_range = []
for page in range(1, pages + 1):
    payload = requests.get(f"{BASE_URL}/api/character",
                           params={"page": page}, timeout=5).json()
    by_range.extend(payload["results"])

print(f"by next link: {len({c['id'] for c in collected})} distinct")
print(f"by range:     {len({c['id'] for c in by_range})} distinct")
print(f"same? {({c['id'] for c in collected}) == ({c['id'] for c in by_range})}")

In [ ]:
# Task 3: exponential backoff
import time

def get_with_retry(url, max_attempts=6, base_delay=0.2):
    delay = base_delay
    for attempt in range(1, max_attempts + 1):
        response = requests.get(url, timeout=10)
        if response.status_code < 400:
            print(f"  attempt {attempt}: {response.status_code} ok")
            return response
        print(f"  attempt {attempt}: {response.status_code}, waiting {delay:.1f}s")
        time.sleep(delay)
        delay *= 2
    raise RuntimeError(f"gave up on {url}")

print(get_with_retry(f"{BASE_URL}/api/flaky").json())

In [ ]:
# Task 4: honour Retry-After
def get_rate_limited(url, max_attempts=8):
    for attempt in range(1, max_attempts + 1):
        response = requests.get(url, timeout=10)
        if response.status_code != 429:
            print(f"  attempt {attempt}: {response.status_code}")
            return response
        wait = float(response.headers.get("Retry-After", 1))
        print(f"  attempt {attempt}: 429, waiting {wait}s as instructed")
        time.sleep(wait)
    raise RuntimeError("still rate limited")

print(get_rate_limited(f"{BASE_URL}/api/limited").json())

In [ ]:
# DE Task 5: fetch_all with guards
def fetch_all(session, start_url, max_pages=50):
    """Follow next links, never revisiting a URL. Returns (records, pages)."""
    records, pages, seen = [], 0, set()
    url = start_url
    while url and pages < max_pages:
        if url in seen:
            print(f"  loop detected at {url}")
            break
        seen.add(url)
        response = session.get(url, timeout=10)
        response.raise_for_status()
        payload = response.json()
        records.extend(payload["results"])
        pages += 1
        url = payload["info"]["next"]
    if pages >= max_pages:
        print(f"  stopped at the {max_pages} page limit")
    return records, pages

session = requests.Session()
rows, pages = fetch_all(session, f"{BASE_URL}/api/character")
print(f"{len(rows)} records over {pages} pages")

In [ ]:
# DE Task 6: api_get, everything combined
def api_get(session, url, max_attempts=5, base_delay=0.2, **kwargs):
    """One request done properly: timeout, retries, backoff, rate limits."""
    delay = base_delay
    kwargs.setdefault("timeout", 10)

    for attempt in range(1, max_attempts + 1):
        try:
            response = session.get(url, **kwargs)

            if response.status_code == 429:
                wait = float(response.headers.get("Retry-After", delay))
                time.sleep(wait)
                continue

            if 400 <= response.status_code < 500:
                response.raise_for_status()          # our fault, do not retry

            if response.status_code >= 500:
                if attempt == max_attempts:
                    response.raise_for_status()
                time.sleep(delay)
                delay *= 2
                continue

            return response

        except (requests.exceptions.ConnectionError, requests.exceptions.Timeout):
            if attempt == max_attempts:
                raise
            time.sleep(delay)
            delay *= 2

    raise RuntimeError(f"gave up on {url} after {max_attempts} attempts")


print(api_get(session, f"{BASE_URL}/api/flaky").json())
try:
    api_get(session, f"{BASE_URL}/api/character", params={"page": 99})
except requests.exceptions.HTTPError as e:
    print(f"404 not retried, raised immediately: {e.response.status_code}")

In [ ]:
# DE Task 7: filtered fetch
alive, _ = fetch_all(session, f"{BASE_URL}/api/character?status=Alive")
everyone, _ = fetch_all(session, f"{BASE_URL}/api/character")

print(f"alive: {len(alive)} of {len(everyone)} "
      f"({len(alive) / len(everyone) * 100:.0f}%)")

## Flattening JSON: tasks

In [ ]:
# Task 1
import pandas as pd

records = requests.get(f"{BASE_URL}/api/character", timeout=5).json()["results"]
naive = pd.DataFrame(records)
print(naive.dtypes)
print()
print(f"origin is a  {type(naive['origin'].iloc[0]).__name__}")
print(f"episode is a {type(naive['episode'].iloc[0]).__name__}")

In [ ]:
# Task 2
flat = pd.json_normalize(records, sep="_")
print(list(flat.columns))

In [ ]:
# Task 3
table = flat[["id", "name", "status", "species", "origin_name"]].copy()
table = table.rename(columns={"origin_name": "origin"})
table["episode_count"] = [len(r["episode"]) for r in records]
table.head()

In [ ]:
# DE Task 4: all pages, flattened
all_records, _ = fetch_all(session, f"{BASE_URL}/api/character")
characters = pd.json_normalize(all_records, sep="_")
characters = characters.rename(columns={"origin_name": "origin"})
characters["episode_count"] = [len(r["episode"]) for r in all_records]

print(characters["status"].value_counts())
print()
print(characters["species"].value_counts())

In [ ]:
# DE Task 5: the exploded child table
episodes = pd.DataFrame([
    {"character_id": r["id"], "episode_url": url}
    for r in all_records
    for url in r["episode"]
])

print(f"characters:         {len(characters)} rows")
print(f"character_episodes: {len(episodes)} rows")
episodes.head()

In [ ]:
# DE Task 6: stream to jsonl
import json, os

def fetch_to_jsonl(session, start_url, path, max_pages=50):
    """Page through an API, writing each record to a JSONL file as it arrives."""
    url, pages, written = start_url, 0, 0
    with open(path, "w", encoding="utf-8") as f:
        while url and pages < max_pages:
            payload = api_get(session, url).json()
            for record in payload["results"]:
                f.write(json.dumps(record) + "\n")
                written += 1
            pages += 1
            url = payload["info"]["next"]
    return pages, written

pages, written = fetch_to_jsonl(session, f"{BASE_URL}/api/character", "characters.jsonl")

with open("characters.jsonl", encoding="utf-8") as f:
    back = [json.loads(line) for line in f]

print(f"wrote {written} over {pages} pages, read back {len(back)}")
print(f"match: {written == len(back)}")

In [ ]:
# DE Task 7: resumable
def fetch_resumable(session, start_url, path, max_pages=50):
    """Append only records whose id is not already in the file."""
    seen = set()
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            for line in f:
                seen.add(json.loads(line)["id"])

    url, pages, added = start_url, 0, 0
    with open(path, "a", encoding="utf-8") as f:
        while url and pages < max_pages:
            payload = api_get(session, url).json()
            for record in payload["results"]:
                if record["id"] not in seen:
                    f.write(json.dumps(record) + "\n")
                    seen.add(record["id"])
                    added += 1
            pages += 1
            url = payload["info"]["next"]
    return added, len(seen)

for run in [1, 2, 3]:
    added, total = fetch_resumable(session, f"{BASE_URL}/api/character", "characters.jsonl")
    print(f"run {run}: added {added}, total {total}")

## Scraping: tasks

In [ ]:
# Task 1
from urllib.robotparser import RobotFileParser

print(requests.get(f"{BASE_URL}/robots.txt", timeout=5).text)

parser = RobotFileParser()
parser.set_url(f"{BASE_URL}/robots.txt")
parser.read()
for path in ["/shop.html", "/admin"]:
    print(f"{path}: {'allowed' if parser.can_fetch('DEC-Bot', BASE_URL + path) else 'DISALLOWED'}")

In [ ]:
# Task 2
from bs4 import BeautifulSoup

html = requests.get(f"{BASE_URL}/shop.html", timeout=5).text
soup = BeautifulSoup(html, "html.parser")

print(soup.title.string)
print(soup.h1.text)

In [ ]:
# Task 3: find_all
for block in soup.find_all("div", class_="product"):
    name = block.find("h2", class_="name").text.strip()
    price = block.find("span", class_="price").text.strip()
    print(f"{name:<16} {price}")

In [ ]:
# Task 4: CSS selectors
names = [tag.text.strip() for tag in soup.select("div.product h2.name")]
prices = [tag.text.strip() for tag in soup.select("div.product span.price")]

for name, price in zip(names, prices):
    print(f"{name:<16} {price}")

In [ ]:
# Task 5: absolute links
from urllib.parse import urljoin

for link in soup.select("a"):
    href = link.get("href")
    print(f"{link.text.strip():<16} {urljoin(BASE_URL, href)}")

In [ ]:
# DE Task 6: products DataFrame
import pandas as pd

products = []
for block in soup.find_all("div", class_="product"):
    products.append({
        "sku": block.get("data-sku"),
        "name": block.find("h2", class_="name").text.strip(),
        "price": float(block.find("span", class_="price").text.strip()),
        "in_stock": "in-stock" in block.find("span", class_="stock").get("class"),
    })

df = pd.DataFrame(products)
print(df)
print()
print(f"in-stock value: {df.loc[df['in_stock'], 'price'].sum():,.0f}")

In [ ]:
# DE Task 7: the table
from io import StringIO

tables = pd.read_html(StringIO(html))
sales = tables[0]
print(sales)
print(f"total revenue: {sales['Revenue'].sum():,}")

In [ ]:
# DE Task 8: defensive scraping
def scrape_products(url, session):
    """Scrape product blocks, surviving missing tags and unparseable prices."""
    response = session.get(url, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    return [parse_product(block) for block in soup.find_all("div", class_="product")]


def parse_product(block):
    """Parse one block into a dict. Missing fields get defaults, never an exception."""
    def text_of(selector, default=""):
        found = block.select_one(selector)
        return found.text.strip() if found else default

    raw_price = text_of("span.price")
    try:
        price = float(raw_price.replace(",", ""))
    except ValueError:
        price = None

    stock_tag = block.select_one("span.stock")
    return {
        "sku": block.get("data-sku", "unknown"),
        "name": text_of("h2.name", "unnamed"),
        "price": price,
        "in_stock": bool(stock_tag) and "in-stock" in stock_tag.get("class", []),
        "rating": text_of("span.rating", "not shown"),
    }


for product in scrape_products(f"{BASE_URL}/shop.html", session):
    print(product)

print()
broken_html = '<div class="product" data-sku="X-1"><h2 class="name">No Price Here</h2></div>'
broken_block = BeautifulSoup(broken_html, "html.parser").find("div", class_="product")
print(parse_product(broken_block))
print("no crash, price is None, defaults everywhere else")

## Mini project: full worked solution

In [ ]:
import requests, time, json, os
import pandas as pd
from datetime import datetime


def make_session():
    """A session with sensible headers set once."""
    session = requests.Session()
    session.headers.update({
        "User-Agent": "DEC-Class-Bot/1.0 (teaching example)",
        "Accept": "application/json",
    })
    return session


def api_get(session, url, max_attempts=5, base_delay=0.2, **kwargs):
    """One request done properly: timeout, retries, backoff, rate limits."""
    delay = base_delay
    kwargs.setdefault("timeout", 10)
    for attempt in range(1, max_attempts + 1):
        try:
            response = session.get(url, **kwargs)
            if response.status_code == 429:
                time.sleep(float(response.headers.get("Retry-After", delay)))
                continue
            if 400 <= response.status_code < 500:
                response.raise_for_status()
            if response.status_code >= 500:
                if attempt == max_attempts:
                    response.raise_for_status()
                time.sleep(delay)
                delay *= 2
                continue
            return response
        except (requests.exceptions.ConnectionError, requests.exceptions.Timeout):
            if attempt == max_attempts:
                raise
            time.sleep(delay)
            delay *= 2
    raise RuntimeError(f"gave up on {url}")


def fetch_all(session, start_url, max_pages=50):
    """Follow next links, never revisiting a URL."""
    records, pages, seen = [], 0, set()
    url = start_url
    while url and pages < max_pages:
        if url in seen:
            break
        seen.add(url)
        payload = api_get(session, url).json()
        records.extend(payload["results"])
        pages += 1
        url = payload["info"]["next"]
    return records, pages


print("session and fetch helpers defined")

In [ ]:
def flatten(records):
    """Return (characters_df, episodes_df) from raw API records."""
    characters = pd.json_normalize(records, sep="_")
    characters = characters.rename(columns={"origin_name": "origin"})
    characters["episode_count"] = [len(r["episode"]) for r in records]
    characters = characters[
        ["id", "name", "status", "species", "origin", "episode_count"]
    ]

    episodes = pd.DataFrame([
        {"character_id": r["id"], "episode_url": url}
        for r in records for url in r["episode"]
    ])
    return characters, episodes


VALID_STATUSES = {"Alive", "Dead", "unknown"}


def validate(df):
    """Split into (clean, rejected). Rejected rows carry a reason."""
    reasons = []
    for _, row in df.iterrows():
        if pd.isna(row["id"]):
            reasons.append("missing id")
        elif not str(row["name"]).strip():
            reasons.append("empty name")
        elif row["status"] not in VALID_STATUSES:
            reasons.append(f"unexpected status {row['status']!r}")
        else:
            reasons.append(None)

    df = df.assign(reason=reasons)
    clean = df[df["reason"].isna()].drop(columns=["reason"]).copy()
    rejected = df[df["reason"].notna()].copy()
    return clean, rejected


print("flatten and validate defined")

In [ ]:
REJECT_LIMIT = 0.10


def save(characters, episodes, rejected, summary):
    """Write every output file."""
    characters.to_csv("characters.csv", index=False)
    episodes.to_csv("character_episodes.csv", index=False)
    rejected.to_csv("rejected.csv", index=False)
    with open("run_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)


def run(status=None, write=True):
    """Extract, flatten, validate, save. Returns the run summary."""
    started = datetime.now().isoformat(timespec="seconds")
    session = make_session()

    url = f"{BASE_URL}/api/character"
    if status:
        url += f"?status={status}"

    records, pages = fetch_all(session, url)
    characters, episodes = flatten(records)
    clean, rejected = validate(characters)

    rate = len(rejected) / len(characters) if len(characters) else 0
    gate_failed = rate > REJECT_LIMIT

    summary = {
        "started_at": started,
        "finished_at": datetime.now().isoformat(timespec="seconds"),
        "filter": status or "none",
        "pages_fetched": pages,
        "records": len(records),
        "distinct_ids": int(characters["id"].nunique()),
        "clean": len(clean),
        "rejected": len(rejected),
        "rejection_rate_pct": round(rate * 100, 1),
        "episode_rows": len(episodes),
        "status": "failed" if gate_failed else "success",
    }

    if write and not gate_failed:
        save(clean, episodes, rejected, summary)

    print("=== API EXTRACTION REPORT ===")
    for key, value in summary.items():
        print(f"{key:<20} {value}")
    if len(records) != summary["distinct_ids"]:
        print("WARNING: records and distinct ids differ. Check pagination.")
    else:
        print("records == distinct ids, pagination is clean")

    return summary


summary = run()

In [ ]:
# Prove it: run twice, the output file must not grow
import pandas as pd

first = len(pd.read_csv("characters.csv"))
run()
second = len(pd.read_csv("characters.csv"))

print()
print(f"characters.csv rows: {first} then {second}")
print("idempotent" if first == second else "NOT idempotent, investigate")

print()
print("filtered run:")
run(status="Alive")

In [ ]:
# Stretch 11: scrape the shop page in the same run
from bs4 import BeautifulSoup

session = make_session()
html = session.get(f"{BASE_URL}/shop.html", timeout=10).text
soup = BeautifulSoup(html, "html.parser")

products = pd.DataFrame([
    {
        "sku": block.get("data-sku"),
        "name": block.find("h2", class_="name").text.strip(),
        "price": float(block.find("span", class_="price").text.strip()),
        "in_stock": "in-stock" in block.find("span", class_="stock").get("class"),
    }
    for block in soup.find_all("div", class_="product")
])

products.to_csv("products.csv", index=False)
print(products)
print(f"saved {len(products)} product(s) alongside the API extract")

In [ ]:
# Clean up
import os
for name in ["characters.jsonl", "characters.csv", "character_episodes.csv",
             "rejected.csv", "run_summary.json", "products.csv"]:
    if os.path.isfile(name):
        os.remove(name)
_server.shutdown()
print("cleaned up, practice API stopped")